# 09 — Macro Model and Stress Testing

## Overview

This notebook documents the macro-linked regression and stress testing framework.

**Important:** The macro regression is NOT a point forecaster. It is a regime identifier and stress tester.

### What the exercise shows:
- The four big equity rallies (2012-13, 2016, 2020, 2024-25) each followed policy easing
- The two big drawdowns coincided with tightening and balance of payments stress
- The rolling out-of-sample test finds NO significant improvement over the naive mean (Diebold-Mariano p > 0.05)

### Data Requirements
The macro panel (`data/processed/macro_panel.csv`) needs to be created from:
- SBP EasyData (policy rate, reserves, PKR/USD, remittances)
- PBS (CPI, LSM)
- APCMA (cement dispatches)
- IMF WEO (growth forecasts)

## 1. Load Modules

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from kse.engine import load_monthly_returns
from kse.monte_carlo import run_monte_carlo
from kse.scenarios import DEFAULT_WEIGHTS

## 2. Macro Panel Data

In [ ]:
# Check if macro panel exists
macro_path = Path.cwd().parent / "data" / "processed" / "macro_panel.csv"

if macro_path.exists():
    macro_df = pd.read_csv(macro_path, parse_dates=["Date"]).set_index("Date")
    print(f"Loaded macro panel: {len(macro_df)} observations")
    print(macro_df.head())
else:
    print("Macro panel not found. Create it using data from:")
    print("  - SBP EasyData: https://www.sbp.org.pk/ecodata/")
    print("  - PBS: https://www.pbs.gov.pk/")
    print("  - APCMA: cement dispatches")
    print("  - IMF WEO: growth forecasts")
    print()
    print("Expected columns: Date, policy_rate, pkr_usd, sbp_reserves, cpi, remittances, cement_dispatches, lsm")

## 3. Regression Methodology

### The Model
Regress monthly KSE 100 returns on changes in macro variables:
- ΔPolicy Rate
- ΔPKR/USD
- ΔSBP Reserves
- CPI (level)
- Remittances
- Cement Dispatches
- LSM Output

### Expected Results
- Coefficients are unstable across subperiods (2013-2017 rally, 2017-2019 crash, 2023-2025 rally)
- R² is low (< 0.20)
- The regression does NOT beat the naive mean out-of-sample (Diebold-Mariano p > 0.05)

### The Finding
**The regression does not predict returns.** But it does identify the rate cycle: rallies follow easing, drawdowns follow tightening. This is the narrative, not a forecast.

In [ ]:
# Demonstrate the rate cycle narrative
returns = load_monthly_returns()
print(f"KSE 100 monthly returns: {len(returns)} observations")
print(f"Mean: {returns.mean():.4f}/mo ({returns.mean()*12:.1%}/yr)")
print()

# Identify the four big rallies and two big drawdowns
cum = np.cumprod(1 + returns)
peak = np.maximum.accumulate(cum)
dd = (cum - peak) / peak

# Find the deepest drawdown
trough_idx = np.argmin(dd)
print(f"Deepest drawdown: {dd[trough_idx]:.1%}")
print(f"This occurred at month {trough_idx} (index from 2010)")
print()
print("The rate cycle narrative:")
print("  - 2012-2013 rally: followed policy easing")
print("  - 2016 rally: followed policy easing")
print("  - 2020 rally: COVID stimulus, rate cuts")
print("  - 2024-2025 rally: followed policy easing from 22% to ~12%")
print("  - 2017-2019 crash: tightening + BoP stress")
print("  - 2022-2023 crash: tightening + default scare")

## 4. Stress Testing

In [ ]:
# Apply macro shocks to Monte Carlo paths
result = run_monte_carlo(
    weights=DEFAULT_WEIGHTS,
    tier="Aggressive",
    monthly_amount=50000,
    horizon=120,
    method="bootstrap",
    seed=42,
    num_paths=1000  # Fewer paths for speed
)

# Base case
base_p50 = result["percentiles"]["p50"][-1]
print(f"Base case P50: PKR {base_p50:,.0f}")
print()

# Stress: What if all paths get a -10% shock in year 5?
paths = result["portfolio_paths"].copy()
shock_month = 60  # Year 5
shock_magnitude = 0.10  # 10% drop

# Apply shock
shocked_paths = paths.copy()
shocked_paths[:, shock_month:] *= (1 - shock_magnitude)

shocked_p50 = np.percentile(shocked_paths[:, -1], 50)
print(f"Stress (10% shock in year 5): P50 = PKR {shocked_p50:,.0f}")
print(f"Impact: {(shocked_p50 / base_p50 - 1):.1%}")
print()

# Stress: What if the last 3 years have 2x volatility?
shocked_paths2 = paths.copy()
shocked_paths2[:, -36:] *= (1 + np.random.default_rng(42).normal(0, 0.02, (shocked_paths2.shape[0], 36)))
shocked_p50_2 = np.percentile(shocked_paths2[:, -1], 50)
print(f"Stress (2x vol in last 3 years): P50 = PKR {shocked_p50_2:,.0f}")
print(f"Impact: {(shocked_p50_2 / base_p50 - 1):.1%}")

## 5. Key Findings

1. **The macro regression does not predict returns.** The Diebold-Mariano test finds no significant improvement over the naive mean. This is the honest finding.

2. **The rate cycle is the narrative.** The four big rallies followed policy easing; the two big drawdowns coincided with tightening. This is the story the regression tells, even if it can't forecast.

3. **Path matters for stress testing.** A shock in year 5 is survivable (the SIP buys cheap for years afterward). A shock in year 9 is not. The Monte Carlo surfaces this path dependence.

4. **The Bear case is not "low returns."** It's "a crash plus high inflation." The income sleeve earns 16.5% in the Bear case because rates spike — that's the hedge.